# P93 — Sistema de hormigas: optimización mediante una colonia de agentes cooperantes

## 1. Título y paper

**Paper:** *Ant System: Optimization by a Colony of Cooperating Agents*  
**Autoría:** Marco Dorigo, Vittorio Maniezzo, Alberto Colorni  
**Año y venue:** 1996 · IEEE Transactions on Systems, Man and Cybernetics, Part B, 26(1), 29–41  
**Nivel:** L2 · **Motor:** `aco`  
**Ficha completa:** [`P93_aco`](../../papers/foundational/P93_aco/README.md)

**Hito:** La solución no está en ningún agente: está en el rastro que dejan en el entorno y que se refuerza y se evapora.

- [doi:10.1109/3477.484436](https://doi.org/10.1109/3477.484436)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: En problemas combinatorios como el del viajante, las heurísticas golosas se quedan atrapadas en decisiones tempranas y no tienen forma de aprender de los intentos anteriores sin una memoria global costosa.
2. Ejecutar una implementación mínima de la propuesta: Agentes simples que construyen soluciones eligiendo el siguiente paso según una combinación de feromona acumulada y heurística local, y que depositan feromona proporcional a la calidad de la solución construida. La evaporación evita el estancamiento.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P90
- P92
- Grassé (1959), estigmergia en termitas


## 4. Intuición

Las hormigas no se dicen nada. Dejan rastro al pasar, y el rastro se refuerza cuando la ruta es buena y se evapora cuando nadie la usa. La solución no vive en ningún individuo: vive en el entorno, y el entorno recuerda.


## 5. Concepto mínimo

```text
Probabilidad de ir de i a j:  ∝ τ(i,j)^α · η(i,j)^β

    τ  feromona acumulada (memoria compartida)
    η  heurística local (1 / distancia)

Tras cada ronda:  τ ← (1 − ρ)·τ  +  Σ aportes
                  evaporar         reforzar
```


## 6. Código explicado

El motor aísla el mecanismo del paper con datos de juguete y salida inspeccionable.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('aco', seed=7)['result']
show(r)

## 7. Predicción antes de ejecutar

1. ¿Encuentra el óptimo la heurística golosa del vecino más cercano?
2. ¿Y el hormiguero?
3. ¿Qué le pasa al rastro con las iteraciones?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('aco', seed=semilla)
    print(f'semilla {semilla:>2} · evidencia principal:')
    for e in r['evidence']:
        print('   +', e)
    break  # determinista: basta una para ver la estructura
for semilla in (1, 7, 42):
    r = run_paper_lab('aco', seed=semilla)['result']
    print(f'semilla {semilla:>2} → claves: {list(r)[:4]}')

## 9. Salida interpretable

El óptimo mide **26,9634**. El vecino más cercano se queda en **29,5826** —un 9,7 % peor, atrapado por una decisión temprana— y el hormiguero llega al óptimo. Y lo que crece no es la cantidad de feromona sino su **concentración**: la mejor arista pasa de destacar 1,35× sobre la media a 2,5×.


## 10. Comentario pedagógico

El mecanismo tiene dos mitades y las dos son necesarias. El refuerzo distingue lo bueno; la evaporación borra lo que dejó de usarse. Sin evaporación el sistema se casa con la primera ruta decente y deja de explorar — es el mismo fallo que la explotación pura en cualquier problema de bandido.


## 11. Error o anti-patrón deliberado

Anti-patrón: usar un hormiguero para un problema que ya tiene un método específico.


In [ ]:
print('Para el viajante existen Lin-Kernighan y Concorde, que resuelven instancias')
print('de decenas de miles de ciudades de forma optima o casi.')
print('El interes de ACO esta en problemas donde NO existe ese metodo especifico.')

## 12. Corrección

Dónde sí aporta, y con qué evidencia:


In [ ]:
r = run_paper_lab('aco', seed=7)['result']
print('optimo por fuerza bruta :', r['optimo_por_fuerza_bruta'])
print('vecino mas cercano      :', r['vecino_mas_cercano'])
print('mejor del hormiguero    :', r['mejor_del_hormiguero'])
for h in r['historia']:
    print('  ', h)

## 13. Desafío guiado

Sigue la columna de concentración del rastro y explica por qué crece aunque la feromona máxima baje.


In [ ]:
r = run_paper_lab('aco', seed=3)['result']
show(r)

## 14. Desafío autónomo

Modela un problema de rutas o de asignación de tu trabajo y aplica un hormiguero. Compara con la heurística golosa que ya uses y documenta cuánta ventaja saca y a qué coste de cómputo.


## 15. Evidencia de aprendizaje

Guarda la comparación entre óptimo, goloso y hormiguero, y tu explicación del papel de la evaporación.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P93_aco/README.md) · evaluación formal: [`assessments/papers/P93_aco.md`](../../assessments/papers/P93_aco.md)


## 16. Cierre

Ya hay formas de buscar sin gradiente. Volvemos a la probabilidad, y a una pregunta de ingeniería: cómo escribir un modelo sin escribir también su algoritmo de inferencia.


## 17. Conexión con el siguiente hito



Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
